In [55]:
%load_ext autoreload
%autoreload 2

import cv2
import numpy as np

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [56]:
def capture(device: cv2.VideoCapture):
    success, img = device.read()

    if not success: return None

    return img.astype(np.float32) / 255.0

def set_brightness(device, value):
    device.set(cv2.CAP_PROP_BRIGHTNESS, value)

def show_img(img, name="Preview", wait=-1):
    cv2.imshow(name, img)
    return cv2.waitKey(wait)

def set_auto_exposure(device, value):
    device.set(cv2.CAP_PROP_AUTO_EXPOSURE, int(value))

left_camera = cv2.VideoCapture(1)
set_auto_exposure(left_camera, False)
set_brightness(left_camera, 0)

right_camera = cv2.VideoCapture(2)
set_auto_exposure(right_camera, False)
set_brightness(right_camera, 0)

refresh_rate = int(1000.0 / 60)

In [61]:
left_brightness = 0
right_brightness = 0

b_delta = 3

while True:
    left_img = capture(left_camera)
    right_img = capture(right_camera)

    if (left_img is None) or (right_img is None): break 

    # preview = np.hstack([left_img, right_img])
    # preview = cv2.resize(preview, (1200, 600))
    # pressed = show_img(preview, wait=refresh_rate

    pressed = show_img(right_img, wait=refresh_rate)

    # Esc = quit
    if pressed == 27: break

    # Brightness
    if pressed == ord('k'): left_brightness = max(0, left_brightness - b_delta)
    elif pressed == ord('l'): left_brightness = min(100, left_brightness + b_delta)
    set_brightness(left_camera, left_brightness)

    if pressed == ord('e'): right_brightness = max(0, right_brightness - b_delta)
    elif pressed == ord('r'): right_brightness = min(100, right_brightness + b_delta)
    set_brightness(right_camera, right_brightness)

cv2.destroyAllWindows()


### Create the two cameras (left and right)

In [ ]:
left_camera.release()
right_camera.release()

cv2.destroyAllWindows()

### 2x camera - live corner detection of the checkerboard

In [ ]:
charBoard = board.Checkerboard(0.04, (12, 11))

winName = "Characterisation"

imgs = []
validCorners = []

rightImgs = []
rightValidCorners = []

while True:
    if leftCamera.IsFrameReady() and rightCamera.IsFrameReady():
        img = leftCamera.Retrieve().rawData
        rightImg = leftCamera.Retrieve().rawData

        debugImg = img.copy()
        debugRightImg = rightImg.copy()

        # Convert to greyscale for board corner finder
        greyImg = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        rightGreyImg = cv2.cvtColor(rightImg, cv2.COLOR_BGR2GRAY)

        corners = charBoard.FindPOIS(greyImg)
        rightCorners = charBoard.FindPOIS(rightGreyImg)

        imgText = "False"
        if corners is not None:
            debugImg = cv2.drawChessboardCorners(debugImg, charBoard.m_POICount, corners, True)

        if rightCorners is not None:
            debugRightImg = cv2.drawChessboardCorners(debugRightImg, charBoard.m_POICount, rightCorners, True)

        # debugImg = cv2.putText(
        #     debugImg, 
        #     f"Found corners: {imgText}",
        #     (0, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 0), 2)
        
        debugUndistortedImg = np.concatenate([debugImg, debugRightImg], axis=1)
        cv2.imshow(winName, debugUndistortedImg)

    keyPress = cv2.waitKey(int(1000.0 / cameraConfig.refreshRate))

    if keyPress == 27: break # Esc: Finished

    elif (corners is not None) and (rightCorners is not None) and keyPress == 32: # Spacebar: capture camera images
        imgs.append(img)
        rightImgs.append(rightImg)
        validCorners.append(corners)
        rightValidCorners.append(rightCorners)

cv2.destroyWindow(winName)

print(f"Total images captured: {len(imgs)}")

Total images captured: 24


### Single camera - live corner detection of the checkerboard

In [7]:
charBoard = board.Checkerboard(1.5, (11, 8))

winName = "Characterisation"

imgs = []
validCorners = []

while True:
    if leftCamera.IsFrameReady():
        img = leftCamera.Retrieve().rawData

        debugImg = img.copy()

        # Convert to greyscale for board corner finder
        greyImg = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        corners = charBoard.FindPOIS(greyImg)

        imgText = "False"
        if corners is not None:
            debugImg = cv2.drawChessboardCorners(debugImg, charBoard.m_POICount, corners, True)

        debugImg = cv2.resize(debugImg, (600, 600))
        cv2.imshow(winName, debugImg)

    keyPress = cv2.waitKey(int(1000.0 / cameraConfig.refreshRate))

    if keyPress == 27: break # Esc: Finished

    elif keyPress == 32: # Spacebar: capture camera images
        imgs.append(img)
        validCorners.append(corners)

cv2.destroyWindow(winName)

print(f"Total images captured: {len(imgs)}")

Total images captured: 0


### Choose whether to save captured images or not

In [14]:
winName = "Check Images"
output = Path("C:\\Users\\psydw2\\OneDrive - The University of Nottingham\\university\\year3\\measurements\\mixed")
ext = ".jpg"

for i in range(len(imgs)):
    debugImg = cv2.drawChessboardCorners(imgs[i], charBoard.m_POICount, validCorners[i], True)
    cv2.imshow(winName, debugImg)

    while True:
        keyPress = cv2.waitKey(0)
        if keyPress == ord("y"): # Save image
            cv2.imwrite(str(output / f"gm{ext}"), image.ToU8(imgs[i]))
            break
        
        if keyPress == ord("n"):
            break

cv2.destroyWindow(winName)

## Load images from disk

In [ ]:
input_dir = Path("C:\\Users\\psydw2\\OneDrive - The University of Nottingham\\university\\year3\\measurements\\mixed")

left


### Calibrate cameras

In [ ]:
leftCameraCoords = np.asarray(validCorners)
rightCameraCoords = np.asarray(rightValidCorners)

boardCoords = charBoard.GetPOICoords()

objectCoords = np.repeat(boardCoords[np.newaxis, ...], len(leftCameraCoords), axis=0)

leftCamera.Characterise(objectCoords, leftCameraCoords)
rightCamera.Characterise(objectCoords, rightCameraCoords)

print(f"Left reprojection error: {leftCamera.visionConfig.reprojErr}")
print(f"Right reprojection error: {rightCamera.visionConfig.reprojErr}")

leftCamera.visionConfig, rightCamera.visionConfig, reprojErr = vision.RefineCharacterisations(
    leftCamera.visionConfig, rightCamera.visionConfig, 
    objectCoords
)

print(f"Joint reprojection error: {reprojErr}")

Left reprojection error: 0.28383436550505936
Right reprojection error: 0.24678275033812255
Joint reprojection error: 0.32165352876796166


### Show undistorted images using calibration

In [ ]:
winName = "Undistort Images"
output = Path("C:\\Users\\psydw2\\Desktop\\Small Camera\\calibration")
ext = ".jpg"

i = 0
while True:
    if i == len(imgs): break

    l = imgs[i]
    r = rightImgs[i]

    leftUndistorted = leftCamera.Undistort(l)
    rightUndistorted = rightCamera.Undistort(r)

    debugDistortedImg = np.concatenate([l, r], axis=1)
    debugUndistortedImg = np.concatenate([leftUndistorted, rightUndistorted], axis=1)
    compareImg = np.concatenate([debugDistortedImg, debugUndistortedImg], axis=0)

    cv2.imshow(winName, compareImg)

    while True:
        keyPress = cv2.waitKey(0)
        if keyPress == ord("y"): # Save image
            cv2.imwrite(str(output / f"leftUndistorted{i}{ext}"), image.ToU8(leftUndistorted))
            cv2.imwrite(str(output / f"rightUndistorted{i}{ext}"), image.ToU8(rightUndistorted))
            i += 1
            break
        elif keyPress == ord("n"):
            i += 1
            break

cv2.destroyWindow(winName)

### Test system

In [ ]:
winName = "Calculate depth map"

while True:
    if leftCamera.IsFrameReady() and rightCamera.IsFrameReady():
        img = image.ToU8(leftCamera.Retrieve().rawData)
        rightImg = image.ToU8(leftCamera.Retrieve().rawData)

        debugImg = np.concatenate([img, rightImg], axis=1)
        cv2.imshow(winName, debugImg)

    keyPressed = cv2.waitKey(int(1000.0 / cameraConfig.refreshRate))
    if keyPressed == 32: # Spacebar, use this image
        break

# cv2.destroyAllWindows(winName)

# Convert to grayscale for stereo matching (color can be used for point cloud coloring)
greyImg = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
rightGreyImg = cv2.cvtColor(rightImg, cv2.COLOR_BGR2GRAY)

# Compute rectification transforms
c1 = leftCamera.visionConfig
c2 = rightCamera.visionConfig

R1, R2, P1, P2, Q, roi1, roi2 = cv2.stereoRectify(
    c1.intrinsicMat, c1.distortMat, c2.intrinsicMat, c2.distortMat, 
    cameraConfig.resolution, c1.rotation, c1.translation,
    alpha=0
)

# Compute rectification maps
leftMap1, leftMap2 = cv2.initUndistortRectifyMap(
    c1.intrinsicMat, c1.distortMat, R1, P1, cameraConfig.resolution, cv2.CV_32FC1
)

rightMap1, rightMap2 = cv2.initUndistortRectifyMap(
    c2.intrinsicMat, c2.distortMat, R2, P2, cameraConfig.resolution, cv2.CV_32FC1
)

leftRectified = cv2.remap(greyImg, leftMap1, leftMap2, cv2.INTER_LINEAR)
rightRectified = cv2.remap(rightGreyImg, rightMap1, rightMap2, cv2.INTER_LINEAR)

# Also rectify color images for colored point cloud
leftRGBRectified = cv2.remap(img, leftMap1, leftMap2, cv2.INTER_LINEAR)

# Create Stereo SGBM matcher - TUNE THESE PARAMETERS FOR YOUR SCENE!
stereo = cv2.StereoSGBM_create(
    minDisparity=0,        # Minimum disparity (usually 0)
    numDisparities=80,     # Search range: must be divisible by 16. Increase for closer objects.
    blockSize=5,           # Matching block size. Odd number between 3-11.
    P1=8 * 3 * 5**2,       # Smoothness penalty 1
    P2=32 * 3 * 5**2,      # Smoothness penalty 2 (usually 4*P1)
    disp12MaxDiff=1,       # Maximum allowed difference in left-right check
    uniquenessRatio=15,    # Margin in percent for uniqueness check
    speckleWindowSize=100, # Maximum size of smooth disparity regions
    speckleRange=2         # Maximum disparity variation within speckle window
)

disparity = stereo.compute(leftRectified, rightRectified)
disparity = disparity.astype(np.float32) / 16.0

# TODO: Create a mask for valid points in disparity map
disparityMask = disparity > disparity.min()


# Reproject disparity map to 3D coordinates
points = cv2.reprojectImageTo3D(disparity, Q)
points = points[disparityMask]


# Apply mask to get valid points and colors
colourMask = np.stack([disparityMask] * 3, axis=-1)
colours = leftRGBRectified[colourMask]
colours = cv2.cvtColor(colours, cv2.COLOR_BGR2RGB).reshape(-1, 3)

# Remove points that are too far or too close (adjust these values for your setup)
# zFilter = (validPoints[:, 2] > 0.1) & (validPoints[:, 2] < 50.0)  # Adjust max distance as needed
# filtered_points = validPoints[zFilter]
# filtered_colors = colours[zFilter]

# Create Open3D point cloud
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(colours / 255.0)

# Remove statistical outliers
cl, ind = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
clean_pcd = pcd.select_by_index(ind)

# Display disparity map for debugging (normalized for visibility)
debugDisparity = cv2.normalize(disparity, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)
cv2.imshow('Disparity Map', debugDisparity)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Display the point cloud
o3d.visualization.draw_geometries([clean_pcd])


### Clean up camera handles

In [6]:
del leftCamera

### Convert Images to Video (Optasensor)

In [ ]:
img_dir = Path("C:\\Users\\psydw2\\Desktop\\Manekin\\")

for i in range(2):
    exp_dir = img_dir / str(i+1)

    imgs = [cv2.imread(str(exp_dir / "a0.png"))]
    h, w, c = imgs[0].shape

    video = cv2.VideoWriter(
        str(exp_dir / f'video{i+1}.avi'), 
        cv2.VideoWriter_fourcc(*'mp4v'), 
        30, (w, h), True)

    j = 1
    while True:
        # if j % 30 == 0: print(f"Parsing frame {j//30}")

        img = cv2.imread(str(exp_dir / f"a{j}.png"))

        if img is None: break
    
        video.write(img)

        j += 1

    video.release()

cv2.destroyAllWindows()

Parsing frame 1
Parsing frame 2
Parsing frame 3
Parsing frame 4
Parsing frame 5
Parsing frame 6
Parsing frame 7
Parsing frame 8
Parsing frame 9
Parsing frame 10
Parsing frame 11
Parsing frame 12
Parsing frame 13
Parsing frame 14
Parsing frame 15
Parsing frame 16
Parsing frame 17
Parsing frame 18
Parsing frame 19
Parsing frame 20
Parsing frame 21
Parsing frame 22
Parsing frame 23
Parsing frame 24
Parsing frame 25
Parsing frame 26
Parsing frame 27
Parsing frame 28
Parsing frame 29
Parsing frame 30
Parsing frame 31
Parsing frame 32
Parsing frame 33
Parsing frame 34
Parsing frame 35
Parsing frame 36
Parsing frame 37
Parsing frame 38
Parsing frame 39
Parsing frame 40
Parsing frame 41
Parsing frame 42
Parsing frame 43
Parsing frame 44
Parsing frame 45
Parsing frame 46
Parsing frame 47
Parsing frame 48
Parsing frame 49
Parsing frame 50
Parsing frame 51
Parsing frame 52
Parsing frame 53
Parsing frame 54
Parsing frame 55
Parsing frame 56
Parsing frame 57
Parsing frame 58
Parsing frame 59
Parsin